# Supervised Pretraining Baseline — CIFAR-10
ResNet-18 trained with cross-entropy on CIFAR-10 labels. The frozen backbone is
then evaluated with the same linear protocol as SimCLR/MoCo, giving the **upper-bound reference**.

**Expected runtime:** ~45 min on T4.  
**Platform:** Kaggle — enable GPU via Settings → Accelerator → GPU T4 x2.

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'No GPU — enable GPU in Kaggle settings.'

In [ ]:
import os, subprocess

REPO_URL   = 'https://github.com/SAlaMusa/IDL.git'
REPO_DIR   = '/kaggle/working/IDL'
OUTPUT_DIR = '/kaggle/working'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
import glob, shutil, csv, datetime

EPOCHS    = 200
CKPT_PATH = os.path.join(OUTPUT_DIR, f'supervised_ep{EPOCHS}.pth.tar')

if os.path.exists(CKPT_PATH):
    print('Checkpoint exists, skipping pretraining.')
else:
    subprocess.run([
        'python', 'supervised_pretrain.py',
        '--epochs', str(EPOCHS), '--batch-size', '256',
        '-j', '2', '--seed', '42', '--fp16-precision',
    ], check=True)
    latest = max(glob.glob('runs/supervised/*.pth.tar'), key=os.path.getmtime)
    shutil.copy2(latest, CKPT_PATH)
    print(f'Checkpoint saved: {CKPT_PATH}')

In [ ]:
RESULTS = os.path.join(OUTPUT_DIR, 'supervised_results.csv')

subprocess.run([
    'python', 'linear_eval.py',
    '--checkpoint', CKPT_PATH, '--dataset', 'cifar10',
    '--arch', 'resnet18', '--epochs', '100',
    '-b', '256', '-j', '2', '--seed', '42',
], check=True)

with open('linear_eval_results.csv') as f:
    best_top1 = float(list(csv.DictReader(f))[-1]['best_top1'])

with open(RESULTS, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['exp_name', 'best_top1', 'checkpoint', 'timestamp'])
    w.writerow(['supervised', f'{best_top1:.2f}', CKPT_PATH,
                datetime.datetime.now().strftime('%Y-%m-%d %H:%M')])

print(f'\nBest Top-1 (linear eval on frozen supervised backbone): {best_top1:.2f}%')
print(f'Results saved to: {RESULTS}')